# drfn: is the CMM negative result about the idea, the sign, or the code?

Three runs, seed 11, same data, same protocol, same parameter budget. Each
changes exactly one thing from the published configuration.

| run | module | entropy term |
|---|---|---|
| `cmm_s11_diffuse` | `cmm.py` as shipped | maximized (as shipped) |
| `cmm_s11_sharpen` | `cmm.py` as shipped | minimized (section 3.6) |
| `cmmspec_s11_sharpen` | protocol section 3.4 | minimized |

All three keep the auxiliary heads, so supervision is held constant.

## Why run 3 exists

`cmm.py` is **not** the CMM of section 3.4. It uses one `root_bank` as both
key and value with no query projection -- `softmax(h B^T) B` -- where
Eqs. (4)-(5) specify `Q = H W_Q` with separate `M_K` and `M_V`, and Eq. (9)
specifies a per-token scalar gate rather than the d-dimensional one in the
code. With tied keys and values the retrieved vector is pulled toward
whichever slot already lies closest to `h`, which biases slots toward surface
similarity instead of abstract roots.

So the architecture the proposal describes has never actually been trained.
Until it has, the thesis cannot distinguish "compositional memory fails at
13.8K pairs" from "this implementation of it fails", and a negative result is
only worth publishing when it is clean.

## Why runs 1 and 2 exist

`train.py` computes `total = total - entropy_reg * ent`, where `ent` is the
Shannon entropy of the routing distribution. Minimizing that **maximizes**
entropy, pushing every token toward uniform assignment over slots.

Section 3.6 of the proposal says the opposite: Eq. (12) adds `+alpha * L_ent`,
and the text reads "Minimizing `L_ent` sharpens each token's routing toward few
slots". Diffuse routing is named there as failure mode (i), "the bottleneck
never binds" -- so as implemented, the regularizer optimizes for the failure.

`README.md` assumption 1 records the minus as a deliberate reconstruction of
the lost original rather than a slip. Either way the published sentence
"unsupervised CMM ... routes roots at chance" (0.337 against a 0.330
permutation baseline) was measured under a term pushing routing toward chance,
so it needs re-deriving before it is defended.

Two runs, identical except for the sign. Compare
`routing_consistency_root` in each `metadata.json`.

In [ ]:
SEED = 11

In [ ]:
!pip install -q tokenizers sacrebleu
import torch; print(torch.__version__, torch.cuda.is_available())

In [ ]:
import glob, os, shutil, subprocess, sys, json
def find(pat):
    hits = sorted(glob.glob('/kaggle/input/**/' + pat, recursive=True))
    if not hits: raise RuntimeError('not found under /kaggle/input: ' + pat)
    return hits[0]
pack_src = os.path.dirname(find('run_grid.sh'))
PACK = '/kaggle/working/pack'
if not os.path.exists(PACK): shutil.copytree(pack_src, PACK)
os.chdir(PACK)
print('pack', pack_src)
def stream(cmd, env=None, skip_steps=True):
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, env=env)
    for line in p.stdout:
        if skip_steps and (line.startswith('{"step"') or line.startswith("{'step'")):
            continue
        print(line, end='')
    return p.wait()

In [ ]:
print('exit', stream(['bash', 'run_entropy_check.sh']))

In [ ]:
# best dev loss, chrF++ at that epoch, and the routing probe
rows = []
for d in sorted(glob.glob('runs/*_s*')):
    lg = os.path.join(d, 'train_log.jsonl')
    md = os.path.join(d, 'metadata.json')
    if not os.path.exists(lg): continue
    ev = [json.loads(l) for l in open(lg) if '"dev_loss"' in l]
    if not ev: continue
    b = min(ev, key=lambda e: e['dev_loss'])
    rc = {}
    if os.path.exists(md):
        rc = (json.load(open(md)).get('routing_consistency_root') or {})
    rows.append((os.path.basename(d), len(ev), b['dev_loss'], b['dev_chrf'],
                 b['epoch'], rc.get('consistency'), rc.get('permutation')))
print('%-22s %6s %10s %8s %5s %8s %8s' %
      ('run', 'epochs', 'dev_loss', 'chrF++', 'best', 'route', 'perm'))
for r in rows:
    print('%-22s %6d %10.4f %8.2f %5d %8s %8s' %
          (r[0], r[1], r[2], r[3], r[4],
           'n/a' if r[5] is None else '%.3f' % r[5],
           'n/a' if r[6] is None else '%.3f' % r[6]))

by = {r[0]: r for r in rows}
chrf = lambda n: by[n][3] if n in by else None
route = lambda n: by[n][5] if n in by else None
S = globals().get('SEED', 11)

need = [f'bpe_aux_s{S}', f'bpe_placebo_s{S}', f'gmm_s{S}', f'gmm_placebo_s{S}']
if all(n in by for n in need):
    head = chrf(need[0]) - chrf(need[1])
    mem = chrf(need[2]) - chrf(need[3])
    print('\n--- the deciding comparison ---')
    print('head   real - placebo : %+.2f chrF++   (published value +0.47)' % head)
    print('memory real - placebo : %+.2f chrF++' % mem)
    print('difference of gaps    : %+.2f' % (mem - head))
    print('\nA memory gap clearly larger than the head gap is the result the')
    print('architecture predicts: only the memory can share one stored vector')
    print('across the surface forms of a root, and the placebo permutes labels')
    print('ACROSS WORDS, which destroys exactly that sharing. Equal gaps mean')
    print('morphological CONTENT adds nothing beyond auxiliary-task')
    print('regularization at this data size -- a controlled negative result,')
    print('which is also worth reporting.')

ent = [f'cmm_s{S}_diffuse', f'cmm_s{S}_sharpen', f'cmmspec_s{S}_sharpen']
if any(n in by for n in ent):
    print('\n--- entropy sign and module fidelity ---')
    print('%-26s %8s %8s %8s' % ('run', 'chrF++', 'route', 'perm'))
    for n, what in zip(ent, ['published cmm.py, entropy MAXIMIZED (as shipped)',
                             'published cmm.py, entropy minimized (section 3.6)',
                             'spec 3.4 module, entropy minimized']):
        if n in by:
            r = by[n]
            print('%-26s %8.2f %8s %8s   <- %s' %
                  (n, r[3], 'n/a' if r[5] is None else '%.3f' % r[5],
                   'n/a' if r[6] is None else '%.3f' % r[6], what))
    print('\nPublished reference: routing consistency 0.337, permutation 0.330.')
    print('If the sharpened run moves off chance, the negative result was')
    print('measured under a term pushing routing toward chance. If only the')
    print('spec module moves, the published cmm.py was the problem. If neither')
    print('moves, the mechanism genuinely fails to bind at 13.8K pairs -- and')
    print('that is now a clean negative result rather than a confounded one.')
print('\nNOTE: greedy chrF++ on 256 dev sentences is noisy. This seed-11 grid')
print('is a go/no-go signal, not the thesis number. The six-seed grid and the')
print('held-out paradigm set decide; see DECISION_RULE.')
open('/kaggle/working/summary.txt', 'w').write(repr(rows))